# Gold: lab_test_trends

Builds `project1_gs_caresync.gold.lab_test_trends` (one row per test_name per calendar month) per the STTM `sttm_silver_to_gold.xlsx`. Driving table: `silver.lab_results` grouped by test_name, year, month; LEFT JOIN to `calendar` on the month's first day; LAG() for month-over-month trend. Full rebuild (CREATE OR REPLACE).

Schema notes: the calendar key column is `datekey` (INT, yyyyMMdd), not `date_key` as written in the STTM.

In [0]:
%sql
CREATE OR REPLACE TABLE project1_gs_caresync.gold.lab_test_trends
COMMENT 'Gold: one row per test type per month with abnormal/critical rates and month-over-month trend (STTM silver->gold)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                                   AS year,
    month(test_date)                                  AS month,
    CAST(date_trunc('MONTH', MIN(test_date)) AS DATE) AS month_start,
    COUNT(*)                                          AS total_tests,
    COUNT_IF(result_status = 'Abnormal')              AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')              AS critical_test_count
  FROM project1_gs_caresync.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
with_lag AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  m.test_name,
  CAST(m.year AS INT)                                AS year,
  CAST(m.month AS INT)                               AS month,
  c.month_name,
  c.quarter_name,
  CAST(m.total_tests AS INT)                         AS total_tests,
  CAST(m.abnormal_test_count AS INT)                 AS abnormal_test_count,
  CAST(m.critical_test_count AS INT)                 AS critical_test_count,
  m.abnormal_rate_pct,
  m.critical_rate_pct,
  m.prev_month_abnormal_rate_pct,
  CAST(m.abnormal_rate_pct - m.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag m
LEFT JOIN project1_gs_caresync.silver.calendar c
  ON c.datekey = CAST(date_format(m.month_start, 'yyyyMMdd') AS INT)

In [0]:
%sql
-- Grain check: (test_name, year, month) unique; total tests reconcile to silver.lab_results
SELECT
  COUNT(*)                                    AS gold_rows,
  COUNT(DISTINCT test_name, year, month)      AS distinct_test_months,
  SUM(total_tests)                            AS sum_total_tests,
  (SELECT COUNT(*) FROM project1_gs_caresync.silver.lab_results) AS silver_lab_rows,
  COUNT_IF(month_name IS NULL)                AS missing_calendar_rows
FROM project1_gs_caresync.gold.lab_test_trends

In [0]:
updated_df = spark.table("project1_gs_caresync.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))

for getting row count in each notebook 

the notebooks are working fine. now you shall return the record count as output in all 3 notebooks using below kind of code 
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))